In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('../data/messy_clinical_appointments.db')
df = pd.read_sql("SELECT * FROM messy_clinic_appointments_original", conn)

In [3]:
df

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required,...,booking_date_day,booking_date_day_padded,booking_date_month,booking_date_month_padded,booking_date_year,booking_date_final,age_bracket,lead_time_days,lead_time_ranges,lead_time_category
0,1080,Tammy Williams,76,female,2/26/26,12/3/24,Christopher Graham,Cardiology,¬£425.8,1,...,3,03,12,12,2024,2024-12-03,Elderly,450.0,0.821822,Long Wait
1,1074,Megan Strickland,69,F,5/23/25,12-Jun-24,Brandon Lewis,Orthopedics,‚Ç¨344.26,Y,...,12,12,Jun,06,2024,2024-06-12,Elderly,345.0,0.659660,Moderate Wait
2,1067,Amanda Schroeder,79,M,30-Nov-25,5-Aug-24,Deanna Edwards,Neurology,‚Ç¨203.34,Y,...,5,05,Aug,08,2024,2024-08-05,Elderly,482.0,0.870871,Long Wait
3,1072,Anthony Mcpherson,47,F,18-May-25,9/9/24,Karen Parsons,General,Rs85.76,No,...,9,09,9,09,2024,2024-09-09,Middle Aged,251.0,0.481481,Standard Wait
4,1092,Benjamin Brown,45,None,3/7/26,8/17/24,Andrea Hernandez,General,$84.44,1,...,17,17,8,08,2024,2024-08-17,Middle Aged,567.0,0.942943,Extreme Wait
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1082,Thomas Roman,64,Female,14-Sep-25,7/15/24,Robert Bass,General,Rs200.43,Yes,...,15,15,7,07,2024,2024-07-15,Middle Aged,426.0,0.793794,Long Wait
996,1008,Jack Cooper,67,F,9-Apr-25,20-Jun-24,Christopher Smith,General,Rs244.49,No,...,20,20,Jun,06,2024,2024-06-20,Elderly,293.0,0.554555,Moderate Wait
997,1022,Christopher Brown,28,1,13-Feb-26,6/13/24,Kimberly Johnson,Neurology,¬£154.48,N,...,13,13,6,06,2024,2024-06-13,Young Adult,610.0,0.965966,Extreme Wait
998,1088,Michael Sullivan,65,F,7/4/25,4/15/24,Anthony Mccoy,Neurology,$434.95,0,...,15,15,4,04,2024,2024-04-15,Elderly,445.0,0.816817,Long Wait


In [4]:
df.columns

Index(['patient_id', 'patient_name', 'age', 'gender', 'appointment_date',
       'booking_date', 'doctor', 'department', 'billing_amount',
       'follow_up_required', 'patient_id_fixed', 'gender_fixed',
       'follow_up_required_fix', 'identify_currency', 'extract_number',
       'billing_amount_fixed', 'appointment_date_day',
       'appointment_date_day_padded', 'appointment_date_month',
       'appointment_date_month_padded', 'appointment_date_year',
       'appointment_date_final', 'booking_date_day', 'booking_date_day_padded',
       'booking_date_month', 'booking_date_month_padded', 'booking_date_year',
       'booking_date_final', 'age_bracket', 'lead_time_days',
       'lead_time_ranges', 'lead_time_category'],
      dtype='object')

In [5]:
df['age'].describe()

count    1000.000000
mean       53.750000
std        21.137604
min        18.000000
25%        34.000000
50%        55.000000
75%        71.000000
max        90.000000
Name: age, dtype: float64

In [6]:
df[['patient_id_fixed','patient_name','age','age_bracket','gender_fixed','department','follow_up_required_fix','billing_amount_fixed','appointment_date_final','booking_date_final','lead_time_category']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   patient_id_fixed        1000 non-null   int64  
 1   patient_name            1000 non-null   object 
 2   age                     1000 non-null   int64  
 3   age_bracket             1000 non-null   object 
 4   gender_fixed            693 non-null    object 
 5   department              1000 non-null   object 
 6   follow_up_required_fix  664 non-null    object 
 7   billing_amount_fixed    950 non-null    float64
 8   appointment_date_final  1000 non-null   object 
 9   booking_date_final      1000 non-null   object 
 10  lead_time_category      1000 non-null   object 
dtypes: float64(1), int64(2), object(8)
memory usage: 86.1+ KB


In [7]:
# drop missing rows that have missing follow_up_required_fix
df = df.dropna(subset=['follow_up_required_fix']).copy()

In [8]:
df[['follow_up_required_fix']].isnull().sum()

follow_up_required_fix    0
dtype: int64

In [9]:
# replacing missing billing_amount_fixed
df['billing_amount_fixed'] = df['billing_amount_fixed'].fillna(df['billing_amount_fixed'].median())

In [10]:
df['billing_amount_fixed']

1      375.24340
2      221.64060
3        1.02912
5      200.42175
6      107.91000
         ...    
994      4.52856
995      2.40516
996      2.93388
997    196.18960
999    111.43070
Name: billing_amount_fixed, Length: 664, dtype: float64

In [11]:
df[['billing_amount_fixed']].isnull().sum()

billing_amount_fixed    0
dtype: int64

In [12]:
# replacing missing gender elements as 'Unknown'
df['gender_fixed'] = df['gender_fixed'].fillna('Unknown')

In [13]:
df['gender_fixed']

1       Female
2         Male
3       Female
5      Unknown
6       Female
        ...   
994       Male
995     Female
996     Female
997    Unknown
999       Male
Name: gender_fixed, Length: 664, dtype: object

In [14]:
df[['gender_fixed']].isnull().sum()

gender_fixed    0
dtype: int64

In [15]:
df = df.drop(columns=['gender', 'appointment_date',
       'booking_date', 'billing_amount',
       'follow_up_required',
       'extract_number',
       'appointment_date_day',
       'appointment_date_day_padded', 'appointment_date_month',
       'appointment_date_month_padded', 'appointment_date_year',
       'booking_date_day', 'booking_date_day_padded',
       'booking_date_month', 'booking_date_month_padded', 'booking_date_year',
       'lead_time_days',
       'lead_time_ranges'])

In [16]:
df

,patient_id,patient_name,age,doctor,department,patient_id_fixed,gender_fixed,follow_up_required_fix,identify_currency,billing_amount_fixed,appointment_date_final,booking_date_final,age_bracket,lead_time_category
1,1074,Megan Strickland,69,Brandon Lewis,Orthopedics,2,Female,Yes,EUR,375.24340,2025-05-23,2024-06-12,Elderly,Moderate Wait
2,1067,Amanda Schroeder,79,Deanna Edwards,Neurology,3,Male,Yes,EUR,221.64060,2025-11-30,2024-08-05,Elderly,Long Wait
3,1072,Anthony Mcpherson,47,Karen Parsons,General,4,Female,No,INR,1.02912,2025-05-18,2024-09-09,Middle Aged,Standard Wait
5,1057,Joe Sharp,78,Kimberly Fields,General,6,Unknown,Yes,None,200.42175,2025-09-26,2025-08-31,Elderly,Standard Wait
6,1018,Alexander Harris,22,Jim Jones,Cardiology,7,Female,No,EUR,107.91000,2025-10-30,2024-12-29,Young Adult,Moderate Wait
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,1068,Aaron Holder,19,Lauren Ayers,General,995,Male,Yes,INR,4.52856,2026-01-26,2024-11-28,Young Adult,Long Wait
995,1082,Thomas Roman,64,Robert Bass,General,996,Female,Yes,INR,2.40516,2025-09-14,2024-07-15,Middle Aged,Long Wait
996,1008,Jack Cooper,67,Christopher Smith,General,997,Female,No,INR,2.93388,2025-04-09,2024-06-20,Elderly,Moderate Wait
997,1022,Christopher Brown,28,Kimberly Johnson,Neurology,998,Unknown,No,GBP,196.18960,2026-02-13,2024-06-13,Young Adult,Extreme Wait


In [17]:
df['patient_id_fixed'] = range(1, len(df) + 1)

In [18]:
df = df.reset_index(drop=True)

In [19]:
df = df.set_index('patient_id_fixed')

In [20]:
# format the entire DataFrame to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# format a specific column for display
df['billing_amount_fixed'] = df['billing_amount_fixed'].round(2)

In [21]:
df.to_csv('../data/cleaned_appointments.csv')

In [22]:
df

,patient_id,patient_name,age,doctor,department,gender_fixed,follow_up_required_fix,identify_currency,billing_amount_fixed,appointment_date_final,booking_date_final,age_bracket,lead_time_category
patient_id_fixed,,,,,,,,,,,,,
1,1074,Megan Strickland,69,Brandon Lewis,Orthopedics,Female,Yes,EUR,375.24,2025-05-23,2024-06-12,Elderly,Moderate Wait
2,1067,Amanda Schroeder,79,Deanna Edwards,Neurology,Male,Yes,EUR,221.64,2025-11-30,2024-08-05,Elderly,Long Wait
3,1072,Anthony Mcpherson,47,Karen Parsons,General,Female,No,INR,1.03,2025-05-18,2024-09-09,Middle Aged,Standard Wait
4,1057,Joe Sharp,78,Kimberly Fields,General,Unknown,Yes,None,200.42,2025-09-26,2025-08-31,Elderly,Standard Wait
5,1018,Alexander Harris,22,Jim Jones,Cardiology,Female,No,EUR,107.91,2025-10-30,2024-12-29,Young Adult,Moderate Wait
...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,1068,Aaron Holder,19,Lauren Ayers,General,Male,Yes,INR,4.53,2026-01-26,2024-11-28,Young Adult,Long Wait
661,1082,Thomas Roman,64,Robert Bass,General,Female,Yes,INR,2.41,2025-09-14,2024-07-15,Middle Aged,Long Wait
662,1008,Jack Cooper,67,Christopher Smith,General,Female,No,INR,2.93,2025-04-09,2024-06-20,Elderly,Moderate Wait
